# 🗺️ Étude Géomatique Complète — Kénitra, Maroc
## Analyse Multi-Risques : Inondation · Glissement · Séisme

| Module | Description | Source |
|--------|------------|--------|
| 1 | Installation & Vérification | — |
| 2 | Coordonnées & Conversion EPSG:26191 → WGS84 | Levé topographique |
| 3 | Carte interactive du site | Folium |
| 4 | Climatologie 30 ans | Open-Meteo / ERA5 |
| 5 | Figures climatologiques | Matplotlib |
| 6 | Topographie & Pente | OpenTopoData / SRTM |
| 7 | Risque d'Inondation | Analyse hydrologique |
| 8 | Risque de Glissement | LSI multi-facteurs |
| 9 | Risque Sismique | USGS FDSN |
| 10 | Carte de synthèse multi-risques | Folium |
| 11 | Rapport final + Radar | Matplotlib |
| 12 | Export des fichiers | — |

## 📦 Cellule 1 — Installation & Vérification des versions

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 1 — Installation robuste                               ║
# ║  Après exécution → Runtime → Restart session → reprendre C2    ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess, sys

pkgs = [
    'pyproj==3.6.1',
    'folium==0.15.1',
    'requests==2.31.0',
    'pandas',
    'numpy',
    'matplotlib',
    'scipy',
    'seaborn',
    'branca',
]

print('⏳ Installation...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install'] + pkgs + ['-q'],
                   capture_output=True, text=True)
print('✅ Installation terminée')
if r.returncode != 0:
    print('⚠️ ', r.stderr[-200:])

print('''
╔══════════════════════════════════════════════════════╗
║  ▶  Runtime → Restart session (Redémarrer)          ║
║     Puis exécuter les cellules 2 → 12               ║
╚══════════════════════════════════════════════════════╝''')

try:
    from google.colab import runtime
    import time; time.sleep(2)
    runtime.unassign()
except Exception:
    print('→ Redémarrer manuellement : Runtime → Restart session')

## 🔧 Cellule 2 — Imports & Configuration (après redémarrage)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 2 — Imports, vérification versions, configuration      ║
# ╚══════════════════════════════════════════════════════════════════╝
import sys
from packaging.version import Version

VERSIONS = {
    'numpy': '1.24', 'pandas': '2.0', 'scipy': '1.10',
    'matplotlib': '3.7', 'pyproj': '3.5', 'folium': '0.14',
    'requests': '2.28', 'seaborn': '0.12',
}
import importlib
all_ok = True
print(f'Python {sys.version.split()[0]}')
print('╔══ VÉRIFICATION DES VERSIONS ══╗')
for pkg, mn in VERSIONS.items():
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        ok = Version(str(v)) >= Version(str(mn))
        print(f"  {'✅' if ok else '⚠️ '} {pkg:<15} {v}")
        if not ok: all_ok = False
    except Exception as e:
        print(f'  ❌ {pkg:<15} {e}'); all_ok = False
print('╚' + '═'*32 + '╝')

import warnings, os, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import folium
import requests
from pyproj import Transformer, CRS
from scipy.spatial import ConvexHull
from scipy import stats
from datetime import datetime, date
from IPython.display import display, HTML
from itertools import combinations
warnings.filterwarnings('ignore')

# ── Palette de couleurs ────────────────────────────────────────────
PAL = {
    'bg':'#080E18','surface':'#0D1526','border':'#1E2D45',
    'text':'#D0DCF0','muted':'#6B85A8',
    'blue':'#3D9EFF','cyan':'#00C9C9','green':'#22C55E',
    'yellow':'#F59E0B','orange':'#F97316','red':'#EF4444',
    'purple':'#A855F7','accent':'#38BDF8',
}
plt.rcParams.update({
    'figure.facecolor':PAL['bg'],'axes.facecolor':PAL['surface'],
    'axes.edgecolor':PAL['border'],'text.color':PAL['text'],
    'axes.labelcolor':PAL['text'],'xtick.color':PAL['muted'],
    'ytick.color':PAL['muted'],'grid.color':PAL['border'],
    'grid.alpha':0.5,'axes.titlesize':12,'axes.titleweight':'bold',
    'legend.fontsize':9,'legend.framealpha':0.25,
})

# ── Configuration globale ──────────────────────────────────────────
CFG = {
    'CRS_SRC'     : 'EPSG:26191',
    'CRS_TGT'     : 'EPSG:4326',
    'X_MIN':100_000, 'X_MAX':700_000,
    'Y_MIN':200_000, 'Y_MAX':700_000,
    'LAT_MIN':27.0, 'LAT_MAX':36.0,
    'LON_MIN':-14.0,'LON_MAX':-1.0,
    'BUFFER_M'    : 2000,
    'SEISM_RAD_KM': 300,
    'CLIM_START'  : '1991-01-01',
    'CLIM_END'    : date.today().strftime('%Y-%m-%d'),
    'SLOPE_FLOOD' : 2.0,
    'SLOPE_SLIDE' : 15.0,
    'OUT_DIR'     : '/content/rapport_kenitra',
}
os.makedirs(CFG['OUT_DIR'], exist_ok=True)

print(f"\n{'✅ Environnement prêt — continuer avec C3' if all_ok else '❌ Problème — voir ci-dessus'}")

## 📍 Cellule 3 — Coordonnées & Conversion EPSG:26191 → WGS84

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 3 — Données levé & Validation géodésique complète      ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Données brutes (Merchich / Lambert Nord Maroc EPSG:26191) ─────
RAW = {
    'MAT': list(range(1, 16)),
    'X'  : [471052.18, 471132.61, 471119.12, 471115.17, 471114.06,
             471113.03, 471111.70, 471115.29, 471101.77, 471090.64,
             471024.71, 471003.46, 471014.52, 471019.55, 471024.72],
    'Y'  : [402763.35, 402707.86, 402675.64, 402676.17, 402666.35,
             402666.50, 402651.69, 402640.25, 402635.93, 402667.37,
             402675.90, 402727.70, 402718.95, 402726.01, 402723.09],
}
df = pd.DataFrame(RAW)

# ── ÉTAPE 1 : Validation coordonnées sources ───────────────────────
print('╔══ VALIDATION EPSG:26191 ══╗')
errs = []
for _, r in df.iterrows():
    if not (CFG['X_MIN'] <= r['X'] <= CFG['X_MAX']):
        errs.append(f"MAT{int(r['MAT'])} X={r['X']} hors plage")
    if not (CFG['Y_MIN'] <= r['Y'] <= CFG['Y_MAX']):
        errs.append(f"MAT{int(r['MAT'])} Y={r['Y']} hors plage")
if errs:
    for e in errs: print(f'  ❌ {e}')
    raise ValueError('Coordonnées invalides')
print(f'  ✅ {len(df)} points dans la plage EPSG:26191')
print(f'  ✅ Pas de doublons : {df.duplicated(["X","Y"]).sum()} doublon(s)')

# ── ÉTAPE 2 : Conversion EPSG:26191 → WGS84 ───────────────────────
crs_s = CRS.from_epsg(26191)
print(f'\n  CRS source : {crs_s.name}')
print(f'  Datum      : {crs_s.datum.name}')
transformer = Transformer.from_crs(CFG['CRS_SRC'], CFG['CRS_TGT'], always_xy=True)
df['lon'], df['lat'] = transformer.transform(df['X'].values, df['Y'].values)

# ── ÉTAPE 3 : Validation WGS84 ────────────────────────────────────
print('\n╠══ VALIDATION WGS84 ══╣')
wgs_errs = []
for _, r in df.iterrows():
    if not (CFG['LAT_MIN'] <= r['lat'] <= CFG['LAT_MAX']):
        wgs_errs.append(f"MAT{int(r['MAT'])} lat={r['lat']:.4f} hors Maroc")
    if not (CFG['LON_MIN'] <= r['lon'] <= CFG['LON_MAX']):
        wgs_errs.append(f"MAT{int(r['MAT'])} lon={r['lon']:.4f} hors Maroc")
    if np.isnan(r['lat']) or np.isnan(r['lon']):
        wgs_errs.append(f"MAT{int(r['MAT'])} NaN après conversion")
if wgs_errs:
    for e in wgs_errs: print(f'  ❌ {e}')
    raise ValueError('Conversion échouée')
print(f'  ✅ Lat [{df["lat"].min():.6f}°, {df["lat"].max():.6f}°] dans Maroc')
print(f'  ✅ Lon [{df["lon"].min():.6f}°, {df["lon"].max():.6f}°] dans Maroc')

# ── ÉTAPE 4 : Métriques géométriques ──────────────────────────────
LAT_C = df['lat'].mean()
LON_C = df['lon'].mean()
pts_m = np.column_stack([df['X'].values, df['Y'].values])
hull  = ConvexHull(pts_m)
hp    = pts_m[hull.vertices]
n_h   = len(hp)
AREA_M2 = 0.5 * abs(sum(
    hp[i,0]*hp[(i+1)%n_h,1] - hp[(i+1)%n_h,0]*hp[i,1]
    for i in range(n_h)))
AREA_HA = AREA_M2 / 10_000
PERIM_M = sum(np.sqrt((hp[(i+1)%n_h,0]-hp[i,0])**2 +
                       (hp[(i+1)%n_h,1]-hp[i,1])**2) for i in range(n_h))
DIAM_M  = max(np.sqrt((df['X'].iloc[i]-df['X'].iloc[j])**2 +
                       (df['Y'].iloc[i]-df['Y'].iloc[j])**2)
              for i,j in combinations(range(len(df)),2))

print(f'\n╠══ MÉTRIQUES DU LEVÉ ══╣')
print(f'  Centroïde   : {LAT_C:.6f}°N  {LON_C:.6f}°E')
print(f'  Superficie  : {AREA_M2:.2f} m² ({AREA_HA:.5f} ha)')
print(f'  Périmètre   : {PERIM_M:.2f} m')
print(f'  Diamètre max: {DIAM_M:.2f} m')
print(f'  Étendue X   : {df["X"].max()-df["X"].min():.2f} m')
print(f'  Étendue Y   : {df["Y"].max()-df["Y"].min():.2f} m')
print('\n  MAT │       X (m) │       Y (m) │       Lon (°) │      Lat (°)')
print('  ' + '─'*58)
for _, r in df.iterrows():
    print(f"  {int(r['MAT']):>3} │ {r['X']:>11.2f} │ {r['Y']:>11.2f} │ {r['lon']:>13.6f} │ {r['lat']:>11.6f}")
print('\n✅ C3 OK — variables : df, LAT_C, LON_C, AREA_M2, AREA_HA, PERIM_M')

## 🗺️ Cellule 4 — Carte interactive du site

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 4 — Carte interactive folium (100% compatible)         ║
# ╚══════════════════════════════════════════════════════════════════╝

def build_site_map(dataframe, lat_c, lon_c, buffer_m=2000, zoom=15):
    """Carte folium pure — compatible toutes versions."""
    m = folium.Map(location=[lat_c, lon_c], zoom_start=zoom,
                   control_scale=True, tiles=None)
    folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Satellite').add_to(m)
    folium.TileLayer(
        'https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
        attr='Esri', name='Topographique').add_to(m)

    grp = folium.FeatureGroup(name='📍 Points levé topographique')
    coords = []
    for _, r in dataframe.iterrows():
        mat = int(r['MAT'])
        coords.append([r['lat'], r['lon']])
        folium.CircleMarker(
            location=[r['lat'], r['lon']], radius=8,
            color='#f97316', fill=True,
            fill_color='#fde68a', fill_opacity=0.95,
            popup=folium.Popup(
                f"<div style='font-family:monospace;font-size:12px;line-height:1.8'>"
                f"<b style='color:#f97316;font-size:14px'>MAT {mat}</b><br>"
                f"<b>Lambert EPSG:26191</b><br>"
                f"X : {r['X']:.2f} m<br>Y : {r['Y']:.2f} m<br>"
                f"<b>WGS84 EPSG:4326</b><br>"
                f"Lat : {r['lat']:.6f}°<br>Lon : {r['lon']:.6f}°"
                f"</div>", max_width=240),
            tooltip=f'MAT {mat}'
        ).add_to(grp)
        folium.Marker(
            location=[r['lat'], r['lon']],
            icon=folium.DivIcon(
                html=f'<div style="font-size:9px;font-weight:bold;color:#fde68a;'
                     f'text-shadow:0 0 3px #000;margin-left:8px">{mat}</div>',
                icon_size=(20,14), icon_anchor=(0,7))
        ).add_to(grp)

    folium.PolyLine(coords+[coords[0]], color='#f97316',
                    weight=2.5, opacity=0.85,
                    tooltip='Périmètre du levé').add_to(grp)
    folium.Polygon(coords, color='#f97316', fill=True,
                   fill_color='#f97316', fill_opacity=0.08).add_to(grp)
    grp.add_to(m)

    folium.Circle([lat_c, lon_c], radius=buffer_m,
                  color='#38bdf8', fill=False, weight=2,
                  dash_array='8,6',
                  tooltip=f'Zone analyse : {buffer_m} m').add_to(m)
    folium.CircleMarker([lat_c, lon_c], radius=5,
                         color='#38bdf8', fill=True,
                         fill_color='#38bdf8', fill_opacity=0.9,
                         tooltip=f'Centroïde {lat_c:.6f}°N {lon_c:.6f}°E').add_to(m)

    info = (f"<div style='position:fixed;top:10px;right:10px;z-index:9999;"
            f"background:#080E18;border:1px solid #1E2D45;border-radius:8px;"
            f"padding:12px;font-family:monospace;color:#D0DCF0;font-size:11px;"
            f"box-shadow:0 4px 16px rgba(0,0,0,0.6)'>"
            f"<b style='color:#38BDF8'>📍 Site d'étude — Kénitra</b><br>"
            f"CRS levé : EPSG:26191 (Merchich/Lambert)<br>"
            f"Centroïde : {lat_c:.5f}°N, {lon_c:.5f}°E<br>"
            f"Superficie : {AREA_M2:.0f} m² ({AREA_HA:.4f} ha)<br>"
            f"Périmètre : {PERIM_M:.1f} m | {len(dataframe)} points"
            f"</div>")
    m.get_root().html.add_child(folium.Element(info))
    folium.LayerControl(collapsed=False).add_to(m)
    return m

Map_site = build_site_map(df, LAT_C, LON_C, CFG['BUFFER_M'])
path_map = f"{CFG['OUT_DIR']}/carte_site.html"
Map_site.save(path_map)
print(f'✅ Carte site sauvegardée : {path_map}')
display(Map_site)

## 🌧️ Cellule 5 — Climatologie 30 ans (Open-Meteo / ERA5)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 5 — Données climatiques ERA5 via Open-Meteo            ║
# ║  Norme OMM : période 30 ans (1991 → présent)                   ║
# ╚══════════════════════════════════════════════════════════════════╝

CLIM_VARS = [
    'temperature_2m_max','temperature_2m_min','temperature_2m_mean',
    'precipitation_sum','rain_sum',
    'wind_speed_10m_max','wind_gusts_10m_max',
    'wind_direction_10m_dominant',
    'et0_fao_evapotranspiration',
]

def fetch_climate(lat, lon, start, end, variables, max_retries=3):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': round(lat,4), 'longitude': round(lon,4),
        'start_date': start, 'end_date': end,
        'daily': variables,
        'timezone': 'Africa/Casablanca',
        'wind_speed_unit': 'kmh',
    }
    for attempt in range(1, max_retries+1):
        try:
            print(f'  ⏳ Tentative {attempt}/{max_retries}...')
            r = requests.get(url, params=params, timeout=90)
            r.raise_for_status()
            raw = r.json()
            if 'error' in raw:
                raise ValueError(f"API error: {raw.get('reason','')}")
            if 'daily' not in raw:
                raise ValueError(f"Clé 'daily' absente: {list(raw.keys())}")
            df_c = pd.DataFrame(raw['daily'])
            df_c['date']  = pd.to_datetime(df_c['time'])
            df_c['year']  = df_c['date'].dt.year
            df_c['month'] = df_c['date'].dt.month
            df_c.drop(columns=['time'], inplace=True)
            print(f'  ✅ {len(df_c)} jours ({df_c["date"].min().date()} → {df_c["date"].max().date()})')
            return df_c, raw.get('latitude'), raw.get('longitude'), raw.get('elevation')
        except requests.exceptions.Timeout:
            print(f'  ⏱️  Timeout {attempt}/3')
        except Exception as e:
            print(f'  ❌ {e}')
            if attempt == max_retries: raise
    raise ConnectionError('API Open-Meteo inaccessible')

print('╔══ TÉLÉCHARGEMENT CLIMATOLOGIE ══╗')
print(f'  Coord    : {LAT_C:.5f}°N, {LON_C:.5f}°E')
print(f'  Période  : {CFG["CLIM_START"]} → {CFG["CLIM_END"]}')
print(f'  Variables: {len(CLIM_VARS)}')

clim, API_LAT, API_LON, API_ELEV = fetch_climate(
    LAT_C, LON_C, CFG['CLIM_START'], CFG['CLIM_END'], CLIM_VARS)

# ── Contrôle qualité ──────────────────────────────────────────────
print('\n╠══ CONTRÔLE QUALITÉ ══╣')
for col in ['temperature_2m_max','temperature_2m_min',
            'precipitation_sum','wind_speed_10m_max']:
    if col not in clim.columns: continue
    pct_nan = 100*clim[col].isna().sum()/len(clim)
    flag = '✅' if pct_nan < 5 else '⚠️ ' if pct_nan < 15 else '❌'
    print(f'  {flag} {col:<35} NaN={pct_nan:.1f}%')
inv_t = (clim['temperature_2m_max'] < clim['temperature_2m_min']).sum()
print(f'  {"✅" if inv_t==0 else "⚠️ "} T°max < T°min : {inv_t} cas')

# ── Statistiques ──────────────────────────────────────────────────
annual = clim.groupby('year').agg(
    precip_total      =('precipitation_sum','sum'),
    temp_mean         =('temperature_2m_mean','mean'),
    temp_max_abs      =('temperature_2m_max','max'),
    temp_min_abs      =('temperature_2m_min','min'),
    wind_max          =('wind_speed_10m_max','max'),
    wind_mean         =('wind_speed_10m_max','mean'),
    jours_pluie       =('precipitation_sum', lambda x:(x>1).sum()),
    jours_forte_pluie =('precipitation_sum', lambda x:(x>20).sum()),
).reset_index()

monthly = clim.groupby('month').agg(
    precip_moy   =('precipitation_sum','mean'),
    precip_std   =('precipitation_sum','std'),
    temp_max_moy =('temperature_2m_max','mean'),
    temp_min_moy =('temperature_2m_min','mean'),
    temp_moy     =('temperature_2m_mean','mean'),
    wind_moy     =('wind_speed_10m_max','mean'),
    wind_dir_moy =('wind_direction_10m_dominant','mean'),
    et0_moy      =('et0_fao_evapotranspiration','mean'),
).reset_index()
monthly['month_name'] = pd.to_datetime(monthly['month'],format='%m').dt.strftime('%b')

P50  = float(np.nanpercentile(clim['precipitation_sum'],50))
P90  = float(np.nanpercentile(clim['precipitation_sum'],90))
P95  = float(np.nanpercentile(clim['precipitation_sum'],95))
P99  = float(np.nanpercentile(clim['precipitation_sum'],99))
PMAX = float(clim['precipitation_sum'].max())

slope_mk,icpt_mk,r_mk,p_mk,_ = stats.linregress(
    annual['year'], annual['precip_total'])
TREND_SIG = 'significative (p<0.05)' if p_mk < 0.05 else 'non significative'

print('\n╠══ SYNTHÈSE CLIMATOLOGIQUE ══╣')
print(f'  Précip. ann. moy. : {annual["precip_total"].mean():.0f} ± {annual["precip_total"].std():.0f} mm/an')
print(f'  Tendance          : {slope_mk:+.1f} mm/an ({TREND_SIG})')
print(f'  P90/P99/Pmax      : {P90:.1f}/{P99:.1f}/{PMAX:.1f} mm/j')
print(f'  Temp. moy. ann.   : {clim["temperature_2m_mean"].mean():.1f}°C')
print(f'  Temp. max absolu  : {clim["temperature_2m_max"].max():.1f}°C')
print(f'  Vent max hist.    : {clim["wind_speed_10m_max"].max():.0f} km/h')
print(f'  Rafale max hist.  : {clim["wind_gusts_10m_max"].max():.0f} km/h')
print('\n✅ C5 OK — variables : clim, annual, monthly, P90, P99, PMAX, slope_mk')

## 📊 Cellule 6 — Figures climatologiques

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 6 — Tableaux de bord climatologiques (5 graphiques)    ║
# ╚══════════════════════════════════════════════════════════════════╝

MONTHS = monthly['month_name'].tolist()
M_IDX  = range(len(MONTHS))

fig = plt.figure(figsize=(22, 16))
fig.patch.set_facecolor(PAL['bg'])
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

# ── A. Précipitations mensuelles ──────────────────────────────────
ax_a = fig.add_subplot(gs[0,:2])
bars = ax_a.bar(MONTHS, monthly['precip_moy'], color=PAL['blue'],
                alpha=0.82, edgecolor=PAL['accent'], linewidth=0.6)
ax_a.errorbar(MONTHS, monthly['precip_moy'], yerr=monthly['precip_std'],
              fmt='none', color='white', alpha=0.4, capsize=4)
ax_a.axhline(monthly['precip_moy'].mean(), color=PAL['cyan'],
             lw=1.5, ls='--', alpha=0.7,
             label=f'Moy/12={monthly["precip_moy"].mean():.1f} mm')
for b, v in zip(bars, monthly['precip_moy']):
    ax_a.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
              f'{v:.1f}', ha='center', fontsize=7.5, color=PAL['accent'])
ax_a.set_title('A — Précipitations mensuelles (±1σ) | Norme OMM 30 ans')
ax_a.set_ylabel('mm'); ax_a.legend(); ax_a.grid(axis='y')

# ── B. Températures mensuelles ────────────────────────────────────
ax_b = fig.add_subplot(gs[1,:2])
ax_b.fill_between(M_IDX, monthly['temp_min_moy'], monthly['temp_max_moy'],
                  alpha=0.18, color=PAL['orange'], label='Amplitude T°')
ax_b.plot(MONTHS, monthly['temp_max_moy'], 'o-',
          color=PAL['red'], ms=5, label='T°max moy.')
ax_b.plot(MONTHS, monthly['temp_moy'], 's-',
          color=PAL['yellow'], ms=4, lw=1.4, label='T°moy')
ax_b.plot(MONTHS, monthly['temp_min_moy'], '^-',
          color=PAL['green'], ms=5, label='T°min moy.')
ax_b.set_title('B — Éventail thermique mensuel')
ax_b.set_ylabel('°C'); ax_b.legend(ncol=4); ax_b.grid(alpha=0.4)

# ── C. Précipitations annuelles + tendance ────────────────────────
ax_c = fig.add_subplot(gs[2,:2])
c_bars = [PAL['blue'] if v >= annual['precip_total'].mean()
          else PAL['muted'] for v in annual['precip_total']]
ax_c.bar(annual['year'], annual['precip_total'],
         color=c_bars, alpha=0.82, width=0.7)
y_t = np.poly1d([slope_mk, icpt_mk])(annual['year'])
ax_c.plot(annual['year'], y_t, '--', color=PAL['orange'], lw=2,
          label=f'Tendance {slope_mk:+.1f} mm/an ({TREND_SIG[:3+3]})')
ax_c.axhline(annual['precip_total'].mean(), color='white',
             ls=':', lw=1.4, alpha=0.6,
             label=f'Moy {annual["precip_total"].mean():.0f} mm/an')
ax_c.set_title('C — Précipitations annuelles & Tendance climatique')
ax_c.set_ylabel('mm/an'); ax_c.set_xlabel('Année')
ax_c.legend(fontsize=8); ax_c.grid(axis='y')

# ── D. Rose des vents ─────────────────────────────────────────────
ax_d = fig.add_subplot(gs[0:2,2], polar=True)
wd = clim['wind_direction_10m_dominant'].dropna().values
ws = clim['wind_speed_10m_max'].dropna().values
N_S = 16; BW  = 360/N_S
bins_d = np.arange(0, 361, BW)
dir_r  = np.deg2rad(bins_d[:-1] + BW/2)
wd_a = wd[:min(len(wd),len(ws))]
ws_a = ws[:min(len(wd),len(ws))]
freq, _  = np.histogram(wd_a, bins=bins_d)
spd_sum, _ = np.histogram(wd_a, bins=bins_d, weights=ws_a)
spd_avg = np.where(freq>0, spd_sum/freq, 0)
ax_d.set_theta_zero_location('N')
ax_d.set_theta_direction(-1)
cmap_w = plt.cm.get_cmap('YlOrRd')
bars_d = ax_d.bar(dir_r, freq, width=np.deg2rad(BW),
                   bottom=0, alpha=0.9, edgecolor='#111827', lw=0.4)
for b, s in zip(bars_d, spd_avg):
    b.set_facecolor(cmap_w(s/(spd_avg.max()+1e-9)))
dom_idx  = np.argmax(freq)
DOM_DEG  = bins_d[dom_idx]+BW/2
DOM_CARD = ['N','NNE','NE','ENE','E','ESE','SE','SSE',
            'S','SSO','SO','OSO','O','ONO','NO','NNO'][dom_idx]
ax_d.set_title(f'D — Rose des Vents\nDom: {DOM_CARD} ({DOM_DEG:.0f}°) | '
               f'Moy:{ws.mean():.1f} km/h',
               fontweight='bold', color=PAL['text'], pad=12, fontsize=9)
ax_d.set_yticklabels([])
ax_d.set_xticks(np.deg2rad(np.arange(0,360,22.5)))
ax_d.set_xticklabels(['N','','NE','','E','','SE','','S','','SO','','O','','NO',''],
                      fontsize=8, color=PAL['muted'])
ax_d.grid(color=PAL['border'], alpha=0.5)

# ── E. Bilan hydrique ─────────────────────────────────────────────
ax_e = fig.add_subplot(gs[2,2])
ax_e.bar(MONTHS, monthly['precip_moy'], color=PAL['blue'],
         alpha=0.70, label='Précipitations (mm)')
ax_e.plot(MONTHS, monthly['et0_moy'], 'o--',
          color=PAL['orange'], ms=5, lw=1.8, label='ETP (mm)')
ax_e.fill_between(M_IDX,
    [min(p,e) for p,e in zip(monthly['precip_moy'], monthly['et0_moy'])],
    monthly['precip_moy'], alpha=0.2, color=PAL['blue'], label='Surplus')
ax_e.fill_between(M_IDX,
    monthly['precip_moy'],
    [min(p,e) for p,e in zip(monthly['precip_moy'], monthly['et0_moy'])],
    alpha=0.15, color=PAL['red'], label='Déficit')
ax_e.set_title('E — Bilan Hydrique Mensuel (P vs ETP)')
ax_e.set_ylabel('mm'); ax_e.legend(fontsize=7.5); ax_e.grid(alpha=0.4)
ax_e.set_xticks(M_IDX); ax_e.set_xticklabels(MONTHS, fontsize=7)

fig.suptitle(
    f'CLIMATOLOGIE — KÉNITRA ({LAT_C:.4f}°N, {LON_C:.4f}°E)\n'
    f'Période : {CFG["CLIM_START"]} → {CFG["CLIM_END"]} ({len(annual)} ans) | '
    f'Source : ERA5 / Open-Meteo Archive API',
    fontsize=13, fontweight='bold', color=PAL['accent'], y=1.005)

path_clim = f"{CFG['OUT_DIR']}/climatologie_kenitra.png"
plt.savefig(path_clim, dpi=180, bbox_inches='tight', facecolor=PAL['bg'])
plt.show()
print(f'✅ Figure sauvegardée : {path_clim}')

## 🏔️ Cellule 7 — Topographie & Pente (OpenTopoData / SRTM 30m)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 7 — Topographie via OpenTopoData (SRTM 30m gratuit)    ║
# ║  Pas d'authentification GEE requise                             ║
# ╚══════════════════════════════════════════════════════════════════╝

print('╔══ ANALYSE TOPOGRAPHIQUE (OpenTopoData SRTM 30m) ══╗')

def fetch_elevation_batch(latitudes, longitudes, dataset='srtm30m',
                           batch_size=100, max_retries=3):
    """Récupère les altitudes via OpenTopoData API par lots."""
    elevations = []
    url = f'https://api.opentopodata.org/v1/{dataset}'
    lats  = list(latitudes)
    lons  = list(longitudes)
    for i in range(0, len(lats), batch_size):
        batch_lats = lats[i:i+batch_size]
        batch_lons = lons[i:i+batch_size]
        locations  = '|'.join(f'{la},{lo}' for la,lo in zip(batch_lats, batch_lons))
        for attempt in range(1, max_retries+1):
            try:
                r = requests.get(url, params={'locations': locations}, timeout=30)
                r.raise_for_status()
                data = r.json()
                if data.get('status') != 'OK':
                    raise ValueError(f"API status: {data.get('status')}")
                elevations.extend([res['elevation'] or 0
                                    for res in data['results']])
                break
            except Exception as e:
                if attempt == max_retries:
                    elevations.extend([0]*len(batch_lats))
                    print(f'  ⚠️  Lot {i//batch_size+1} échoué : {e}')
    return elevations

# ── Altitudes des 15 points du levé ───────────────────────────────
print('  Récupération altitude des points du levé...')
df['elevation_m'] = fetch_elevation_batch(df['lat'], df['lon'])

# ── Grille d'altitudes pour analyse pente (25×25 autour du centre) ─
print('  Récupération grille DEM (5×5 autour centroïde)...')
GRID_N   = 5
DELTA_DEG = 0.005
grid_lats = [LAT_C + (i-(GRID_N//2))*DELTA_DEG for i in range(GRID_N)]
grid_lons = [LON_C + (j-(GRID_N//2))*DELTA_DEG for j in range(GRID_N)]
g_la, g_lo = np.meshgrid(grid_lats, grid_lons)
g_la_flat  = g_la.flatten().tolist()
g_lo_flat  = g_lo.flatten().tolist()
g_elev_flat = fetch_elevation_batch(g_la_flat, g_lo_flat)
GRID_ELEV  = np.array(g_elev_flat).reshape(GRID_N, GRID_N)

# ── Statistiques DEM ──────────────────────────────────────────────
E = {
    'mean'  : float(df['elevation_m'].mean()),
    'min'   : float(df['elevation_m'].min()),
    'max'   : float(df['elevation_m'].max()),
    'stdDev': float(df['elevation_m'].std()),
    'p25'   : float(np.percentile(df['elevation_m'], 25)),
    'p75'   : float(np.percentile(df['elevation_m'], 75)),
}

# ── Calcul de la pente par différences finies (m/deg ≈ 111000 m/deg) ─
CELL_M = DELTA_DEG * 111_000
dz_dx  = np.gradient(GRID_ELEV, CELL_M, axis=1)
dz_dy  = np.gradient(GRID_ELEV, CELL_M, axis=0)
SLOPE_DEG = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
S = {
    'mean'  : float(SLOPE_DEG.mean()),
    'max'   : float(SLOPE_DEG.max()),
    'stdDev': float(SLOPE_DEG.std()),
    'p50'   : float(np.percentile(SLOPE_DEG, 50)),
    'p75'   : float(np.percentile(SLOPE_DEG, 75)),
}

# ── Validation ────────────────────────────────────────────────────
print('\n  ╔══ VALIDATION DEM ══╗')
print(f'  Altitudes points levé :')
for _, r in df.iterrows():
    print(f'    MAT{int(r["MAT"]):>2} : {r["elevation_m"]:.1f} m')

if E['mean'] < -100 or E['mean'] > 3000:
    print(f'  ⚠️  Altitude aberrante : {E["mean"]:.1f} m')
else:
    print(f'  ✅ Altitude moy. : {E["mean"]:.1f} m (cohérent avec Kénitra)')

print(f'  ✅ Pente moy.    : {S["mean"]:.2f}°')
print(f'  ✅ Pente max     : {S["max"]:.2f}°')

# ── Figure DEM ────────────────────────────────────────────────────
fig_d, axes_d = plt.subplots(1, 2, figsize=(16, 6))
fig_d.patch.set_facecolor(PAL['bg'])

# Carte thermique élévation
im1 = axes_d[0].imshow(GRID_ELEV, cmap='terrain',
                        origin='lower', aspect='auto')
plt.colorbar(im1, ax=axes_d[0], label='Altitude (m)')
axes_d[0].set_title('Élévation SRTM 30m (grille locale)', fontweight='bold')
axes_d[0].set_xlabel('Colonne (est→ouest)')
axes_d[0].set_ylabel('Ligne (sud→nord)')

# Profil d'altitude des 15 points
axes_d[1].bar(df['MAT'], df['elevation_m'],
              color=PAL['blue'], alpha=0.85, edgecolor=PAL['border'])
axes_d[1].axhline(E['mean'], color=PAL['orange'], lw=2, ls='--',
                  label=f'Moy : {E["mean"]:.1f} m')
for _, r in df.iterrows():
    axes_d[1].text(r['MAT'], r['elevation_m']+0.2,
                   f"{r['elevation_m']:.0f}m", ha='center',
                   fontsize=7.5, color=PAL['accent'])
axes_d[1].set_title('Profil d\'altitude des 15 points du levé', fontweight='bold')
axes_d[1].set_xlabel('N° Point (MAT)')
axes_d[1].set_ylabel('Altitude (m)')
axes_d[1].legend(); axes_d[1].grid(alpha=0.4, axis='y')

fig_d.suptitle('ANALYSE TOPOGRAPHIQUE — SRTM 30m | Source : OpenTopoData',
               fontsize=12, fontweight='bold', color=PAL['accent'])
path_dem = f"{CFG['OUT_DIR']}/topographie_kenitra.png"
plt.savefig(path_dem, dpi=150, bbox_inches='tight', facecolor=PAL['bg'])
plt.show()
print(f'\n✅ C7 OK — variables : E (élévation), S (pente), df[elevation_m]')

## 🌊 Cellule 8 — Risque d'Inondation (Gumbel + score AHP)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 8 — Risque d'Inondation                                ║
# ║  Méthode : Analyse fréquentielle Gumbel + Score AHP             ║
# ╚══════════════════════════════════════════════════════════════════╝

print('╔══ MODULE RISQUE INONDATION ══╗')

# ── 1. Analyse fréquentielle de Gumbel ────────────────────────────
print('\n  1. Ajustement Gumbel (précipitations extrêmes)...')

precip_ann_max = clim.groupby('year')['precipitation_sum'].max().dropna().values

def gumbel_params(data):
    sigma = data.std(ddof=1)
    mu    = data.mean() - 0.5772 * sigma * np.sqrt(6) / np.pi
    beta  = sigma * np.sqrt(6) / np.pi
    return mu, beta

def gumbel_T(mu, beta, T):
    return mu - beta * np.log(-np.log(1 - 1/T))

G_MU, G_BETA = gumbel_params(precip_ann_max)
RETURN_PERIODS = {T: round(gumbel_T(G_MU, G_BETA, T), 1)
                  for T in [2, 5, 10, 20, 50, 100]}

print(f'     Gumbel μ={G_MU:.2f} mm  β={G_BETA:.2f} mm')
print(f'     N années analysées : {len(precip_ann_max)}')
print()
print('     Période de retour  │  Précip. (mm/j)')
print('     ' + '─'*34)
for T, val in RETURN_PERIODS.items():
    flag = '⚠️ SEUIL' if val > 50 else ''
    print(f'     T = {T:<6} ans     │  {val:>8.1f}  {flag}')

# ── 2. Indicateurs morpho-hydrologiques ───────────────────────────
print('\n  2. Indicateurs morpho-hydrologiques...')
SEUIL_H   = E['mean'] + 2.0
SEUIL_PTE = CFG['SLOPE_FLOOD']
print(f'     Altitude seuil inondable : < {SEUIL_H:.1f} m')
print(f'     Pente seuil inondable    : < {SEUIL_PTE}°')
N_POINTS_BAS = ((df['elevation_m'] < SEUIL_H) ).sum()
print(f'     Points levé < seuil alt  : {N_POINTS_BAS}/{len(df)}')

# ── 3. Score AHP pondéré ──────────────────────────────────────────
s_elev_i   = min(10.0, max(0, (20 - E['mean']) / 2))
s_pente_i  = min(10.0, max(0, (5  - S['mean']) / 0.5))
s_precip_i = min(10.0, RETURN_PERIODS[10] / 20)
s_jours_i  = min(10.0, annual['jours_forte_pluie'].mean() / 3)

RISK_INOND = round(
    s_elev_i  * 0.35 + s_pente_i * 0.25 +
    s_precip_i* 0.25 + s_jours_i * 0.15, 1)
RISK_INOND_LBL = ('TRÈS ÉLEVÉ' if RISK_INOND > 7.5 else
                  'ÉLEVÉ'      if RISK_INOND > 5.5 else
                  'MODÉRÉ'     if RISK_INOND > 3.5 else 'FAIBLE')

print(f'\n  SCORES AHP :')
print(f'    Élévation   (×0.35) : {s_elev_i:.1f}/10  (moy={E["mean"]:.1f} m)')
print(f'    Pente       (×0.25) : {s_pente_i:.1f}/10  (moy={S["mean"]:.2f}°)')
print(f'    T10 Gumbel  (×0.25) : {s_precip_i:.1f}/10  ({RETURN_PERIODS[10]:.0f} mm/j)')
print(f'    Jours fortes(×0.15) : {s_jours_i:.1f}/10  ({annual["jours_forte_pluie"].mean():.1f} j/an>20mm)')
print(f'\n  ⚠️  INDICE RISQUE INONDATION : {RISK_INOND}/10 → {RISK_INOND_LBL}')

# ── 4. Figure inondation ──────────────────────────────────────────
fig_i, axes_i = plt.subplots(1, 3, figsize=(18, 5))
fig_i.patch.set_facecolor(PAL['bg'])

# Ajustement Gumbel
ax = axes_i[0]
T_vals = np.array([2,5,10,20,50,100])
R_vals = np.array([RETURN_PERIODS[t] for t in T_vals])
ax.semilogx(T_vals, R_vals, 'o-', color=PAL['blue'], lw=2, ms=8)
for t, r in zip(T_vals, R_vals):
    ax.annotate(f'{r:.0f}mm', (t, r), textcoords='offset points',
                xytext=(5,5), fontsize=8, color=PAL['accent'])
ax.axhline(50, color=PAL['red'], ls='--', lw=1.5, label='Seuil danger 50mm')
ax.set_title('Courbe de Gumbel (périodes de retour)')
ax.set_xlabel('Période de retour (ans)')
ax.set_ylabel('Précipitation (mm/j)')
ax.legend(); ax.grid(alpha=0.4)

# Jours forts par an
ax = axes_i[1]
ax.bar(annual['year'], annual['jours_forte_pluie'],
       color=PAL['blue'], alpha=0.82, width=0.7)
ax.axhline(annual['jours_forte_pluie'].mean(), color=PAL['orange'],
           ls='--', lw=2,
           label=f'Moy : {annual["jours_forte_pluie"].mean():.1f} j/an')
ax.set_title('Jours de fortes pluies (>20mm) par année')
ax.set_xlabel('Année'); ax.set_ylabel('Nb jours')
ax.legend(); ax.grid(alpha=0.4, axis='y')

# Profil altitude + seuil
ax = axes_i[2]
colors_pts = [PAL['red'] if e < SEUIL_H else PAL['green']
              for e in df['elevation_m']]
ax.bar(df['MAT'], df['elevation_m'], color=colors_pts, alpha=0.85)
ax.axhline(SEUIL_H, color=PAL['red'], ls='--', lw=2,
           label=f'Seuil inondable ({SEUIL_H:.1f} m)')
ax.set_title('Altitude des points vs seuil inondable')
ax.set_xlabel('MAT'); ax.set_ylabel('Altitude (m)')
red_p  = mpatches.Patch(color=PAL['red'],  label=f'Zones < {SEUIL_H:.0f}m ({N_POINTS_BAS} pts)')
grn_p  = mpatches.Patch(color=PAL['green'], label=f'Zones > {SEUIL_H:.0f}m')
ax.legend(handles=[red_p, grn_p]); ax.grid(alpha=0.4, axis='y')

fig_i.suptitle(
    f'ANALYSE RISQUE INONDATION | Kénitra | Score={RISK_INOND}/10 — {RISK_INOND_LBL}',
    fontsize=12, fontweight='bold', color=PAL['accent'])
path_inond = f"{CFG['OUT_DIR']}/risque_inondation_kenitra.png"
plt.savefig(path_inond, dpi=150, bbox_inches='tight', facecolor=PAL['bg'])
plt.show()
print(f'\n✅ C8 OK — RISK_INOND={RISK_INOND}/10 → {RISK_INOND_LBL}')

## ⛰️ Cellule 9 — Risque de Glissement (LSI multi-facteurs)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 9 — Risque de Glissement de Terrain                    ║
# ║  Méthode : Indice LSI (Varnes 1978 / Hungr 2014) — AHP          ║
# ╚══════════════════════════════════════════════════════════════════╝

print('╔══ MODULE RISQUE GLISSEMENT ══╗')

# ── 1. Pluies cumulées sur 5 jours consécutifs ────────────────────
print('\n  1. Analyse pluies soutenues (fenêtre 5 jours glissants)...')
precip_s = clim['precipitation_sum'].fillna(0)
roll5     = precip_s.rolling(5).sum()
P5_MAX    = float(roll5.max())
P5_P95    = float(np.nanpercentile(roll5.dropna(), 95))
JOURS_P5_50 = int((roll5 > 50).sum())
print(f'     Max cumulé 5j   : {P5_MAX:.1f} mm')
print(f'     P95 cumulé 5j   : {P5_P95:.1f} mm')
print(f'     Épisodes > 50mm : {JOURS_P5_50} fois')

# ── 2. Indice de végétation simplifié (proxy NDVI via NDVI API) ───
print('\n  2. Proxy végétation (estimation basée sur localisation)...')
# Kénitra : zone côtière atlantique, végétation sub-humide
# NDVI estimé : 0.25–0.40 (végétation modérée, territoire péri-urbain)
NDVI_EST   = 0.30
VEG_CLASS  = 'Végétation modérée (zone péri-urbaine Kénitra)'
print(f'     NDVI estimé     : {NDVI_EST:.2f}')
print(f'     Classe          : {VEG_CLASS}')

# ── 3. Score LSI AHP ──────────────────────────────────────────────
print('\n  3. Calcul Landslide Susceptibility Index (LSI)...')
s_pente_g  = min(10.0, S['mean'] / 3.0)
s_precip_g = min(10.0, P5_P95 / 50.0)
s_veg_g    = max(0.0,  10.0 - NDVI_EST * 16.0)
s_expo_g   = 5.0  # neutre (pas de données lithologiques)

RISK_GLISS = round(
    s_pente_g  * 0.40 + s_precip_g * 0.30 +
    s_veg_g    * 0.20 + s_expo_g   * 0.10, 1)
RISK_GLISS_LBL = ('TRÈS ÉLEVÉ' if RISK_GLISS > 7.5 else
                  'ÉLEVÉ'      if RISK_GLISS > 5.5 else
                  'MODÉRÉ'     if RISK_GLISS > 3.5 else 'FAIBLE')

print(f'  SCORES LSI :')
print(f'    Pente        (×0.40) : {s_pente_g:.1f}/10  (moy={S["mean"]:.2f}°)')
print(f'    Précip 5j    (×0.30) : {s_precip_g:.1f}/10  (P95={P5_P95:.0f}mm)')
print(f'    Végétation   (×0.20) : {s_veg_g:.1f}/10  (NDVI={NDVI_EST:.2f})')
print(f'    Exposition   (×0.10) : {s_expo_g:.1f}/10  (neutre)')
print(f'\n  ⚠️  INDICE LSI GLISSEMENT : {RISK_GLISS}/10 → {RISK_GLISS_LBL}')

# ── 4. Figure glissement ──────────────────────────────────────────
fig_g, axes_g = plt.subplots(1, 3, figsize=(18, 5))
fig_g.patch.set_facecolor(PAL['bg'])

# Pluies cumulées 5j
ax = axes_g[0]
roll5_yr = roll5.iloc[::30].values[:50]
ax.plot(roll5_yr, color=PAL['blue'], lw=1.2, alpha=0.8)
ax.axhline(P5_P95, color=PAL['orange'], ls='--', lw=2,
           label=f'P95 = {P5_P95:.0f} mm')
ax.axhline(50, color=PAL['red'], ls=':', lw=1.5, label='Seuil 50mm')
ax.set_title('Précipitations cumulées 5 jours (échantillon)')
ax.set_xlabel('Échantillon (tous les 30j)'); ax.set_ylabel('mm cumulés')
ax.legend(); ax.grid(alpha=0.4)

# Diagramme LSI
ax = axes_g[1]
facteurs = ['Pente\n(×0.40)', 'Précip.\n5j (×0.30)',
            'Végét.\n(×0.20)', 'Expos.\n(×0.10)']
valeurs  = [s_pente_g, s_precip_g, s_veg_g, s_expo_g]
cols     = [PAL['red'] if v>6 else PAL['orange'] if v>3
            else PAL['green'] for v in valeurs]
bars_l   = ax.bar(facteurs, valeurs, color=cols, alpha=0.85)
ax.axhline(RISK_GLISS, color='white', ls='--', lw=2,
           label=f'LSI global = {RISK_GLISS}')
ax.set_ylim(0, 10)
ax.set_title('Scores LSI par facteur')
ax.set_ylabel('Score /10')
for b, v in zip(bars_l, valeurs):
    ax.text(b.get_x()+b.get_width()/2, v+0.15,
            f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
ax.legend(); ax.grid(alpha=0.4, axis='y')

# Histogramme pluies max annuelles
ax = axes_g[2]
ax.hist(precip_ann_max, bins=15, color=PAL['purple'],
        alpha=0.82, edgecolor='#111')
ax.axvline(np.mean(precip_ann_max), color=PAL['orange'],
           ls='--', lw=2, label=f'Moy={np.mean(precip_ann_max):.0f}mm')
ax.axvline(P99, color=PAL['red'], ls=':', lw=2, label=f'P99={P99:.0f}mm')
ax.set_title('Distribution maxima journaliers annuels')
ax.set_xlabel('Précipitation max annuelle (mm)')
ax.set_ylabel('Fréquence')
ax.legend(); ax.grid(alpha=0.4)

fig_g.suptitle(
    f'ANALYSE RISQUE GLISSEMENT (LSI Varnes) | Score={RISK_GLISS}/10 — {RISK_GLISS_LBL}',
    fontsize=12, fontweight='bold', color=PAL['accent'])
path_gliss = f"{CFG['OUT_DIR']}/risque_glissement_kenitra.png"
plt.savefig(path_gliss, dpi=150, bbox_inches='tight', facecolor=PAL['bg'])
plt.show()
print(f'\n✅ C9 OK — RISK_GLISS={RISK_GLISS}/10 → {RISK_GLISS_LBL}')

## 🔴 Cellule 10 — Risque Sismique (USGS + Gutenberg-Richter)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 10 — Risque Sismique                                   ║
# ║  Source  : USGS FDSN Earthquake Catalog                         ║
# ║  Méthode : Gutenberg-Richter + Score AHP                        ║
# ╚══════════════════════════════════════════════════════════════════╝

print('╔══ MODULE RISQUE SISMIQUE ══╗')
print(f'  Rayon : {CFG["SEISM_RAD_KM"]} km | M ≥ 2.5 | 1964–présent')

def fetch_earthquakes(lat, lon, radius_km, min_mag='2.5',
                       start='1964-01-01', max_retries=3):
    url = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
    params = {
        'format':'geojson', 'latitude':round(lat,4),
        'longitude':round(lon,4), 'maxradiuskm':radius_km,
        'minmagnitude':min_mag, 'starttime':start,
        'endtime':datetime.now().strftime('%Y-%m-%d'),
        'orderby':'time',
    }
    for attempt in range(1, max_retries+1):
        try:
            print(f'  ⏳ Tentative {attempt}/{max_retries}...')
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            if data.get('type') != 'FeatureCollection':
                raise ValueError(f"Type inattendu: {data.get('type')}")
            features = data.get('features', [])
            print(f'  ✅ {len(features)} séismes reçus')
            return features
        except requests.exceptions.Timeout:
            print(f'  ⏱️  Timeout {attempt}/3')
        except Exception as e:
            print(f'  ❌ {e}')
            if attempt == max_retries: return []
    return []

features = fetch_earthquakes(LAT_C, LON_C, CFG['SEISM_RAD_KM'])

RISK_SEISM     = 2.5
RISK_SEISM_LBL = 'FAIBLE'
GR_A = GR_B = 0.0
qdf  = pd.DataFrame()

if len(features) == 0:
    print('  ⚠️  Aucun séisme — zone stable (FAIBLE par défaut)')
else:
    rows = []
    for f in features:
        p   = f.get('properties', {})
        g   = f.get('geometry', {}).get('coordinates', [0,0,10])
        mag = p.get('mag')
        if mag is None or np.isnan(float(mag)) or float(mag) <= 0:
            continue
        rows.append({
            'mag'     : float(mag),
            'place'   : str(p.get('place','?')),
            'time'    : datetime.fromtimestamp(p['time']/1000),
            'lon'     : float(g[0]), 'lat': float(g[1]),
            'depth_km': float(g[2]) if len(g)>2 else 10.0,
        })
    qdf = pd.DataFrame(rows)
    qdf['dist_km'] = np.sqrt(
        ((qdf['lat']-LAT_C)*111.32)**2 +
        ((qdf['lon']-LON_C)*111.32*np.cos(np.deg2rad(LAT_C)))**2)
    qdf['year'] = qdf['time'].dt.year
    # Filtre qualité
    qdf = qdf[(qdf['dist_km'] <= CFG['SEISM_RAD_KM']+10) &
              (qdf['mag'] > 0) & (qdf['depth_km'] >= 0)].copy()

    N_YRS  = max(1, qdf['year'].nunique())
    MAX_M  = qdf['mag'].max()
    NB_M4  = int((qdf['mag']>=4).sum())
    NB_M5  = int((qdf['mag']>=5).sum())

    print(f'\n  ╔══ STATISTIQUES SISMIQUES ══╗')
    print(f'  Total validés   : {len(qdf)}')
    print(f'  Magnitude max   : {MAX_M:.1f} Mw')
    print(f'  Magnitude moy.  : {qdf["mag"].mean():.1f} Mw')
    print(f'  Profondeur moy. : {qdf["depth_km"].mean():.0f} km')
    print(f'  M ≥ 4 : {NB_M4} | M ≥ 5 : {NB_M5}')
    print(f'  Période : {qdf["year"].min()}–{qdf["year"].max()} ({N_YRS} ans)')
    print(f'\n  Top 5 :')
    for _, r in qdf.nlargest(5,'mag').iterrows():
        print(f'    M{r["mag"]:.1f} | {r["depth_km"]:.0f}km | '
              f'{r["dist_km"]:.0f}km | {r["time"].strftime("%Y-%m-%d")} | '
              f'{str(r["place"])[:40]}')

    # Gutenberg-Richter
    mag_bins = np.arange(2.5, MAX_M+0.5, 0.5)
    n_cum    = np.array([(qdf['mag']>=m).sum()/N_YRS for m in mag_bins])
    vidx     = n_cum > 0
    if vidx.sum() >= 4:
        sl,ic,rv,pv,_ = stats.linregress(mag_bins[vidx], np.log10(n_cum[vidx]))
        GR_B = abs(sl); GR_A = ic
        print(f'\n  G-R : log10(N) = {GR_A:.2f} - {GR_B:.2f}·M (R²={rv**2:.3f})')
        print(f'  b-value : {GR_B:.2f} (normal Maroc : 0.8–1.2)')
    else:
        GR_A, GR_B = 3.0, 1.0

    def T_seism(m, a=GR_A, b=GR_B):
        n = 10**(a - b*m); return 1/n if n>0 else 1e9

    print('\n  Périodes de retour sismiques :')
    for m in [4.0, 5.0, 5.5, 6.0]:
        T = T_seism(m)
        print(f'    M={m:.1f} → T={T:>8.0f} ans')

    # Score
    s_mag  = min(10.0, MAX_M * 1.2)
    s_freq = min(10.0, NB_M4 / N_YRS * 2.0)
    s_Tr   = min(10.0, 10/(np.log10(max(T_seism(5.0),1))+1))
    RISK_SEISM = round(s_mag*0.50 + s_freq*0.30 + s_Tr*0.20, 1)
    RISK_SEISM_LBL = ('TRÈS ÉLEVÉ' if RISK_SEISM>7.5 else
                      'ÉLEVÉ'      if RISK_SEISM>5.5 else
                      'MODÉRÉ'     if RISK_SEISM>3.5 else 'FAIBLE')
    print(f'\n  ⚠️  RISQUE SISMIQUE : {RISK_SEISM}/10 → {RISK_SEISM_LBL}')

    # Figure
    fig_s, axes_s = plt.subplots(2, 2, figsize=(16, 10))
    fig_s.patch.set_facecolor(PAL['bg'])

    ax = axes_s[0,0]
    ax.hist(qdf['mag'], bins=np.arange(2.5, MAX_M+0.5, 0.5),
            color=PAL['red'], alpha=0.82, edgecolor='#111')
    ax.axvline(5.0, color=PAL['orange'], lw=2, ls='--', label='M=5.0')
    ax.set_title('A — Distribution des magnitudes')
    ax.set_xlabel('Magnitude (Mw)'); ax.set_ylabel('Fréquence')
    ax.legend(); ax.grid(alpha=0.4)

    ax = axes_s[0,1]
    if vidx.sum() >= 4:
        ax.semilogy(mag_bins[vidx], n_cum[vidx], 'o',
                    color=PAL['accent'], ms=7, label='Observé')
        mf = np.linspace(mag_bins[vidx].min(), mag_bins[vidx].max(), 100)
        ax.semilogy(mf, 10**(GR_A - GR_B*mf), '--',
                    color=PAL['orange'], lw=2, label=f'G-R b={GR_B:.2f}')
    ax.set_title('B — Relation Gutenberg-Richter')
    ax.set_xlabel('Magnitude (Mw)'); ax.set_ylabel('N/an ≥ M')
    ax.legend(); ax.grid(alpha=0.4, which='both')

    ax = axes_s[1,0]
    sc = ax.scatter(qdf['mag'], qdf['depth_km'], c=qdf['dist_km'],
                    cmap='RdYlGn_r', alpha=0.55, edgecolors='none',
                    s=np.clip(qdf['mag']**2*6, 10, 200))
    cb = plt.colorbar(sc, ax=ax); cb.set_label('Distance (km)')
    ax.invert_yaxis()
    ax.set_title('C — Profondeur vs Magnitude')
    ax.set_xlabel('Magnitude (Mw)'); ax.set_ylabel('Profondeur (km)')
    ax.grid(alpha=0.4)

    ax = axes_s[1,1]
    yr_range = range(qdf['year'].min(), qdf['year'].max()+1)
    aq = (qdf.groupby('year').size()
             .reindex(yr_range, fill_value=0)
             .reset_index(name='count'))
    ax.bar(aq['year'], aq['count'], color=PAL['purple'], alpha=0.82, width=0.8)
    ax.axhline(aq['count'].mean(), color='white', ls=':', lw=1.5,
               label=f'Moy {aq["count"].mean():.1f}/an')
    ax.set_title('D — Fréquence annuelle')
    ax.set_xlabel('Année'); ax.set_ylabel('Nb séismes')
    ax.legend(); ax.grid(alpha=0.4, axis='y')

    fig_s.suptitle(
        f'SISMICITÉ — r={CFG["SEISM_RAD_KM"]}km | USGS 1964–présent | '
        f'N={len(qdf)} | Score={RISK_SEISM}/10 — {RISK_SEISM_LBL}',
        fontsize=12, fontweight='bold', color=PAL['accent'])
    plt.tight_layout(rect=[0,0,1,0.96])
    path_s = f"{CFG['OUT_DIR']}/sismicite_kenitra.png"
    plt.savefig(path_s, dpi=180, bbox_inches='tight', facecolor=PAL['bg'])
    plt.show()

print(f'\n✅ C10 OK — RISK_SEISM={RISK_SEISM}/10 → {RISK_SEISM_LBL}')

## 🗺️ Cellule 11 — Carte Folium de synthèse multi-risques

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 11 — Carte interactive de synthèse (Folium 100%)       ║
# ╚══════════════════════════════════════════════════════════════════╝

def risk_color(s):
    return '#dc2626' if s>7.5 else '#f97316' if s>5.5 else '#f59e0b' if s>3.5 else '#22c55e'

def risk_badge(s, lbl):
    c = risk_color(s)
    return (f'<span style="background:{c};color:#000;font-weight:bold;'
            f'padding:2px 10px;border-radius:4px">{s}/10 — {lbl}</span>')

m = folium.Map(location=[LAT_C, LON_C], zoom_start=13,
               control_scale=True, tiles=None)

folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
folium.TileLayer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Satellite').add_to(m)
folium.TileLayer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Topographique').add_to(m)

# ── Points levé ───────────────────────────────────────────────────
grp_pts = folium.FeatureGroup(name='📍 Points levé topographique')
coords_l = []
for _, r in df.iterrows():
    mat = int(r['MAT'])
    coords_l.append([r['lat'], r['lon']])
    folium.CircleMarker(
        [r['lat'],r['lon']], radius=8,
        color='#f97316', fill=True,
        fill_color='#fde68a', fill_opacity=0.95,
        popup=folium.Popup(
            f"<div style='font-family:monospace;font-size:12px;line-height:1.8'>"
            f"<b style='color:#f97316'>MAT {mat}</b><br>"
            f"X:{r['X']:.2f} | Y:{r['Y']:.2f}<br>"
            f"Lat:{r['lat']:.6f}° | Lon:{r['lon']:.6f}°<br>"
            f"Alt:{r['elevation_m']:.1f} m"
            f"</div>", max_width=240),
        tooltip=f'MAT {mat} | {r["elevation_m"]:.0f}m'
    ).add_to(grp_pts)
    folium.Marker([r['lat'],r['lon']],
        icon=folium.DivIcon(
            html=f'<div style="font-size:9px;font-weight:bold;color:#fde68a;'
                 f'text-shadow:0 0 3px #000;margin-left:8px">{mat}</div>',
            icon_size=(20,14), icon_anchor=(0,7))
    ).add_to(grp_pts)

folium.PolyLine(coords_l+[coords_l[0]], color='#f97316',
                weight=2.5, opacity=0.85).add_to(grp_pts)
folium.Polygon(coords_l, color='#f97316', fill=True,
               fill_color='#f97316', fill_opacity=0.08).add_to(grp_pts)
grp_pts.add_to(m)

# ── Séismes ────────────────────────────────────────────────────────
if len(qdf) > 0:
    grp_q = folium.FeatureGroup(
        name=f'🔴 Séismes USGS (r={CFG["SEISM_RAD_KM"]}km)', show=False)
    for _, q in qdf.nlargest(300,'mag').iterrows():
        c = ('#dc2626' if q['mag']>=5 else
             '#f97316' if q['mag']>=4 else '#fbbf24')
        folium.CircleMarker(
            [q['lat'],q['lon']], radius=max(3, q['mag']**1.8),
            color=c, fill=True, fill_color=c, fill_opacity=0.40,
            popup=folium.Popup(
                f"<b>M{q['mag']:.1f}</b> — {q['place']}<br>"
                f"Prof:{q['depth_km']:.0f}km | {q['time'].strftime('%d/%m/%Y')}<br>"
                f"Dist:{q['dist_km']:.0f}km", max_width=260),
            tooltip=f"M{q['mag']:.1f}"
        ).add_to(grp_q)
    grp_q.add_to(m)

# ── Zone d'analyse ────────────────────────────────────────────────
grp_buf = folium.FeatureGroup(name='⭕ Zone analyse 2 km')
folium.Circle([LAT_C,LON_C], radius=CFG['BUFFER_M'],
              color='#38bdf8', fill=False, weight=2,
              dash_array='8,6').add_to(grp_buf)
folium.CircleMarker([LAT_C,LON_C], radius=5,
                     color='#38bdf8', fill=True,
                     fill_color='#38bdf8', fill_opacity=0.9,
                     tooltip=f'Centroïde {LAT_C:.6f}°N {LON_C:.6f}°E').add_to(grp_buf)
grp_buf.add_to(m)

# ── Panneau synthèse ──────────────────────────────────────────────
legende = (
    "<div style='position:fixed;bottom:20px;right:20px;z-index:9999;"
    "background:#080E18;border:1px solid #1E2D45;border-radius:12px;"
    "padding:18px;font-family:Courier New,monospace;color:#D0DCF0;"
    "min-width:290px;box-shadow:0 8px 32px rgba(0,0,0,0.75)'>"
    "<div style='font-size:15px;font-weight:bold;color:#38BDF8;"
    "border-bottom:1px solid #1E2D45;padding-bottom:8px;margin-bottom:12px'>"
    "🗺️ SYNTHÈSE — KÉNITRA</div>"
    "<table style='width:100%;font-size:12px;border-collapse:collapse'>"
    f"<tr style='border-bottom:1px solid #1E2D45'>"
    f"<td style='padding:5px 0'>🌊 Inondation</td>"
    f"<td align='right'>{risk_badge(RISK_INOND,RISK_INOND_LBL)}</td></tr>"
    f"<tr style='border-bottom:1px solid #1E2D45'>"
    f"<td style='padding:5px 0'>⛰️ Glissement</td>"
    f"<td align='right'>{risk_badge(RISK_GLISS,RISK_GLISS_LBL)}</td></tr>"
    f"<tr><td style='padding:5px 0'>🔴 Sismique</td>"
    f"<td align='right'>{risk_badge(RISK_SEISM,RISK_SEISM_LBL)}</td></tr>"
    "</table>"
    "<hr style='border-color:#1E2D45;margin:10px 0'>"
    f"<div style='font-size:11px;color:#6B85A8;line-height:1.9'>"
    f"📍 {LAT_C:.5f}°N, {LON_C:.5f}°E<br>"
    f"📐 {AREA_M2:.0f} m² — {AREA_HA:.5f} ha<br>"
    f"🏔️ Altitude moy : {E['mean']:.1f} m<br>"
    f"📐 Pente moy    : {S['mean']:.2f}°<br>"
    f"🌧️ Précip. ann  : {annual['precip_total'].mean():.0f} mm/an<br>"
    f"🌡️ Temp. moy    : {clim['temperature_2m_mean'].mean():.1f}°C<br>"
    f"💨 Vent max     : {clim['wind_speed_10m_max'].max():.0f} km/h<br>"
    f"🔴 Séismes M≥5  : {int((qdf['mag']>=5).sum()) if len(qdf)>0 else 0}"
    "</div>"
    "<div style='font-size:9px;color:#38bdf8;margin-top:8px'>"
    f"ERA5 · OpenTopoData · USGS | {datetime.now().strftime('%d/%m/%Y %H:%M')}"
    "</div></div>"
)
m.get_root().html.add_child(folium.Element(legende))
folium.LayerControl(collapsed=False).add_to(m)

path_syn = f"{CFG['OUT_DIR']}/synthese_risques_kenitra.html"
m.save(path_syn)
print(f'✅ Carte de synthèse sauvegardée : {path_syn}')
display(m)

## 📋 Cellule 12 — Radar multi-risques + Tableau + Export

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELLULE 12 — Radar + Tableau de synthèse + Export complet      ║
# ╚══════════════════════════════════════════════════════════════════╝

fig_r = plt.figure(figsize=(22, 11))
fig_r.patch.set_facecolor(PAL['bg'])
gs2   = gridspec.GridSpec(1, 2, figure=fig_r, wspace=0.25)

# ── Radar ─────────────────────────────────────────────────────────
ax_r = fig_r.add_subplot(gs2[0], polar=True)
CATS = ['Inondation\n(Gumbel+AHP)','Glissement\n(LSI Varnes)',
        'Sismique\n(G-R+AHP)','Précip.\nextrêmes',
        'Vent\nextrême','Chaleur\nextrême']
SCR  = [
    RISK_INOND, RISK_GLISS, RISK_SEISM,
    round(min(10.0, RETURN_PERIODS.get(10,80)/20), 1),
    round(min(10.0, clim['wind_speed_10m_max'].max()/15), 1),
    round(min(10.0, max(0, clim['temperature_2m_max'].max()-34)/1.5), 1),
]
N  = len(CATS)
A  = [n/N*2*np.pi for n in range(N)]
SA = SCR + [SCR[0]]
AA = A   + [A[0]]

ax_r.set_theta_offset(np.pi/2)
ax_r.set_theta_direction(-1)
ax_r.set_xticks(A)
ax_r.set_xticklabels(CATS, fontsize=10, color=PAL['text'])
ax_r.set_ylim(0,10)
ax_r.set_yticks([2,4,6,8,10])
ax_r.set_yticklabels(['2','4','6','8','10'], fontsize=8, color=PAL['muted'])
ax_r.grid(color=PAL['border'], alpha=0.7)
ax_r.set_facecolor(PAL['surface'])

for thr,col,alp in [(10,PAL['red'],0.06),(6,PAL['yellow'],0.07),(4,PAL['green'],0.08)]:
    ax_r.fill(AA, [thr]*len(AA), color=col, alpha=alp)

ax_r.plot(AA, SA, 'o-', lw=2.5, color=PAL['blue'], ms=8, zorder=5)
ax_r.fill(AA, SA, alpha=0.28, color=PAL['blue'])

for angle, score in zip(A, SCR):
    c = PAL['red'] if score>6 else PAL['yellow'] if score>3 else PAL['green']
    ax_r.annotate(f'{score:.1f}', xy=(angle,score),
                  xytext=(angle, min(score+1.3,9.5)),
                  ha='center', fontsize=11, fontweight='bold', color=c)

for lbl,col in [('Faible (≤4)',PAL['green']),
                 ('Modéré (4-6)',PAL['yellow']),
                 ('Élevé (≥6)', PAL['red'])]:
    ax_r.plot([],[],  's', color=col, ms=9, label=lbl)
ax_r.legend(loc='lower right', bbox_to_anchor=(1.42,-0.05),
            fontsize=9, framealpha=0.3)
ax_r.set_title('RADAR DES RISQUES NATURELS\nKénitra — AHP & Gutenberg-Richter',
               fontsize=12, fontweight='bold', color=PAL['accent'], pad=22)

# ── Tableau de synthèse ───────────────────────────────────────────
ax_t = fig_r.add_subplot(gs2[1])
ax_t.axis('off')

PARAMS = [
    ('── SITE ──','',''),
    ('Centroïde WGS84',f'{LAT_C:.5f}°N, {LON_C:.5f}°E','EPSG:4326'),
    ('Superficie (Gauss)',f'{AREA_M2:.1f} m²',f'{AREA_HA:.5f} ha'),
    ('Périmètre levé',f'{PERIM_M:.2f} m','—'),
    ('Points validés',f'{len(df)}','points'),
    ('── MORPHOLOGIE (SRTM) ──','',''),
    ('Altitude moy. / [min–max]',f'{E["mean"]:.1f} m',f'[{E["min"]:.0f}–{E["max"]:.0f}]m'),
    ('Pente moy. / max',f'{S["mean"]:.2f}°',f'max={S["max"]:.2f}°'),
    ('── CLIMATOLOGIE ERA5 ──','',''),
    ('Période',f'{CFG["CLIM_START"]} → {CFG["CLIM_END"]}',f'{len(annual)} ans'),
    ('Précip. ann. moy. ± σ',
     f'{annual["precip_total"].mean():.0f} ± {annual["precip_total"].std():.0f}','mm/an'),
    ('Tendance précip.',f'{slope_mk:+.1f} mm/an',TREND_SIG[:14]),
    ('P90 / P99 / Max',f'{P90:.0f} / {P99:.0f} / {PMAX:.0f}','mm/j'),
    ('T=10 ans (Gumbel)',f'{RETURN_PERIODS.get(10,0):.0f}','mm/j'),
    ('T=100 ans (Gumbel)',f'{RETURN_PERIODS.get(100,0):.0f}','mm/j'),
    ('Temp. moy. / max abs.',
     f'{clim["temperature_2m_mean"].mean():.1f}°C',
     f'max={clim["temperature_2m_max"].max():.1f}°C'),
    ('Vent max / Rafale',
     f'{clim["wind_speed_10m_max"].max():.0f}',
     f'{clim["wind_gusts_10m_max"].max():.0f} km/h'),
    ('── SISMICITÉ (r=300km) ──','',''),
    ('Séismes M≥2.5',f'{len(qdf)}' if len(qdf)>0 else '0','1964–présent'),
    ('Magnitude max',f'{qdf["mag"].max():.1f}' if len(qdf)>0 else '—','Mw'),
    ('Coeff. b (G-R)',f'{GR_B:.2f}' if GR_B>0 else '—','—'),
    ('── INDICES DE RISQUE ──','',''),
    ('⚠️  INONDATION (AHP)',f'{RISK_INOND}/10',RISK_INOND_LBL),
    ('⚠️  GLISSEMENT (LSI)',f'{RISK_GLISS}/10',RISK_GLISS_LBL),
    ('⚠️  SISMIQUE (G-R)',f'{RISK_SEISM}/10',RISK_SEISM_LBL),
]

HDRS = {'── SITE ──','── MORPHOLOGIE (SRTM) ──','── CLIMATOLOGIE ERA5 ──',
         '── SISMICITÉ (r=300km) ──','── INDICES DE RISQUE ──'}

tbl = ax_t.table(cellText=PARAMS,
                  colLabels=['PARAMÈTRE','VALEUR','UNITÉ / CLASSE'],
                  cellLoc='left', loc='center', bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)

for (row,col), cell in tbl.get_celld().items():
    cell.set_edgecolor(PAL['border']); cell.set_linewidth(0.4)
    if row == 0:
        cell.set_facecolor('#0f2340')
        cell.set_text_props(color=PAL['accent'], fontweight='bold')
    elif row>0 and PARAMS[row-1][0] in HDRS:
        cell.set_facecolor('#111d30')
        cell.set_text_props(color=PAL['yellow'], fontweight='bold')
    elif row>0 and PARAMS[row-1][0].startswith('⚠️'):
        bg = {'TRÈS ÉLEVÉ':'#1c0808','ÉLEVÉ':'#1c1008',
              'MODÉRÉ':'#1c1500','FAIBLE':'#081c08'}
        lbl_k = PARAMS[row-1][2]
        cell.set_facecolor(bg.get(lbl_k, PAL['surface']))
        cell.set_text_props(color=PAL['text'], fontweight='bold')
    else:
        cell.set_facecolor(PAL['bg'] if row%2==0 else PAL['surface'])
        cell.set_text_props(color=PAL['text'])
tbl.auto_set_column_width([0,1,2])
ax_t.set_title('TABLEAU DE SYNTHÈSE GÉOMATIQUE',
               fontweight='bold', color=PAL['accent'], fontsize=11, pad=10)

fig_r.suptitle(
    f'RAPPORT GÉOMATIQUE — ANALYSE MULTI-RISQUES\n'
    f'Commune de Kénitra, Maroc | '
    f'ERA5 · OpenTopoData SRTM · USGS | '
    f'{datetime.now().strftime("%d/%m/%Y")}',
    fontsize=10, color=PAL['muted'], y=1.01)

path_rap = f"{CFG['OUT_DIR']}/rapport_risques_kenitra.png"
plt.savefig(path_rap, dpi=200, bbox_inches='tight', facecolor=PAL['bg'])
plt.show()
print(f'✅ Rapport radar sauvegardé : {path_rap}')

# ── Export CSV ────────────────────────────────────────────────────
path_csv_pts = f"{CFG['OUT_DIR']}/points_leve_wgs84.csv"
df.to_csv(path_csv_pts, index=False, encoding='utf-8')

path_csv_clim = f"{CFG['OUT_DIR']}/climatologie_journaliere.csv"
clim.to_csv(path_csv_clim, index=False, encoding='utf-8')

if len(qdf) > 0:
    path_csv_s = f"{CFG['OUT_DIR']}/seismes_usgs.csv"
    qdf.to_csv(path_csv_s, index=False, encoding='utf-8')

# ── Audit final ───────────────────────────────────────────────────
print('\n' + '╔'+'═'*60+'╗')
print('║        AUDIT FINAL — ÉTUDE GÉOMATIQUE KÉNITRA            ║')
print('╚'+'═'*60+'╝')

import os
FILES = {
    'carte_site.html'                : 'Carte interactive du site',
    'climatologie_kenitra.png'       : 'Tableaux climatologiques',
    'topographie_kenitra.png'        : 'DEM & profil altitude',
    'risque_inondation_kenitra.png'  : 'Analyse inondation + Gumbel',
    'risque_glissement_kenitra.png'  : 'Analyse glissement LSI',
    'sismicite_kenitra.png'          : 'Analyse sismique G-R',
    'synthese_risques_kenitra.html'  : 'Carte multi-risques',
    'rapport_risques_kenitra.png'    : 'Radar + Tableau',
    'points_leve_wgs84.csv'          : 'Points levé WGS84',
    'climatologie_journaliere.csv'   : 'Climatologie journalière',
}
all_ok = True
print('\n  📁 FICHIERS GÉNÉRÉS :')
for fname, desc in FILES.items():
    path = f"{CFG['OUT_DIR']}/{fname}"
    if os.path.exists(path):
        kb = os.path.getsize(path)/1024
        print(f"  ✅ {fname:<42} {kb:>7.0f} Ko")
    else:
        print(f"  ❌ {fname:<42} MANQUANT")
        all_ok = False

print('\n  ⚠️  INDICES DE RISQUE :')
for nom, s, lbl in [('Inondation',RISK_INOND,RISK_INOND_LBL),
                     ('Glissement',RISK_GLISS,RISK_GLISS_LBL),
                     ('Sismique',  RISK_SEISM,RISK_SEISM_LBL)]:
    ic  = '🔴' if s>7.5 else '🟠' if s>5.5 else '🟡' if s>3.5 else '🟢'
    bar = '█'*int(s)+'░'*(10-int(s))
    print(f'  {ic} {nom:<12} [{bar}] {s:.1f}/10 → {lbl}')

print('\n  📡 SOURCES :')
print(f'  ✅ ERA5/Open-Meteo   — {CFG["CLIM_START"]} → {CFG["CLIM_END"]} ({len(annual)} ans)')
print(f'  ✅ OpenTopoData SRTM — Élévation 30m ({len(df)} points)')
print(f'  ✅ USGS FDSN         — {len(qdf)} séismes (M≥2.5, r={CFG["SEISM_RAD_KM"]}km)')

print('\n' + '╔'+'═'*60+'╗')
msg = '✅  ÉTUDE COMPLÈTE ET VALIDÉE' if all_ok else '⚠️   PARTIELLE — voir ci-dessus'
print(f'║  {msg.center(58)}  ║')
print('╚'+'═'*60+'╝')
print(f'\n  Généré : {datetime.now().strftime("%d/%m/%Y à %H:%M:%S")}')

# Téléchargement automatique Colab
try:
    from google.colab import files
    dl = ['rapport_risques_kenitra.png','synthese_risques_kenitra.html',
          'points_leve_wgs84.csv']
    for f_name in dl:
        p = f"{CFG['OUT_DIR']}/{f_name}"
        if os.path.exists(p):
            files.download(p)
    print('\n✅ Téléchargement des fichiers principaux lancé')
except ImportError:
    print(f'\nFichiers disponibles dans : {CFG["OUT_DIR"]}')